# Cleaning Notebooks with notebookx

This notebook demonstrates how to clean notebooks using `notebookx`:

- Removing outputs and execution counts
- Using preset cleaning options
- Cleaning notebooks for version control

## Setup

First, let's import the necessary modules:

In [ ]:
import json
from notebookx import Notebook, Format, CleanOptions

## Loading a Notebook with Outputs

Let's load a notebook that has been executed and contains outputs:

In [ ]:
# Load a notebook with outputs
nb = Notebook.from_file("../../nb_format_examples/World population.ipynb")
print(f"Original notebook: {len(nb)} cells")

## Inspecting the Original State

Let's check how many outputs and execution counts are present in the notebook:

In [ ]:
# Check original state (serialize to inspect)
original_json = json.loads(nb.to_string(Format.Ipynb))
code_cells = [c for c in original_json["cells"] if c["cell_type"] == "code"]
outputs_count = sum(len(c.get("outputs", [])) for c in code_cells)
exec_counts = sum(1 for c in code_cells if c.get("execution_count") is not None)

print(f"  - Cells with outputs: {sum(1 for c in code_cells if c.get('outputs'))}")
print(f"  - Total outputs: {outputs_count}")
print(f"  - Cells with execution count: {exec_counts}")

## Cleaning for Version Control

The `CleanOptions.for_vcs()` preset removes outputs and execution counts, which is ideal for version control. This makes notebooks easier to diff and reduces repository size.

In [ ]:
# Clean for version control (removes outputs and execution counts)
clean_nb = nb.clean(CleanOptions.for_vcs())

## Verifying the Cleaning

Let's verify that the outputs and execution counts have been removed:

In [ ]:
# Verify cleaning
clean_json = json.loads(clean_nb.to_string(Format.Ipynb))
clean_code_cells = [c for c in clean_json["cells"] if c["cell_type"] == "code"]
clean_outputs = sum(len(c.get("outputs", [])) for c in clean_code_cells)
clean_exec = sum(1 for c in clean_code_cells if c.get("execution_count") is not None)

print(f"After cleaning (for_vcs):")
print(f"  - Total outputs: {clean_outputs}")
print(f"  - Cells with execution count: {clean_exec}")

## Custom Cleaning Options

You can also create custom cleaning options to control exactly what gets removed:

In [ ]:
# Custom cleaning options
custom_options = CleanOptions(
    remove_outputs=True,
    remove_execution_counts=True,
    remove_cell_metadata=True,
)
custom_clean = nb.clean(custom_options)
print(f"Custom cleaning also works: {len(custom_clean)} cells")

## Strip All: Maximum Cleaning

The `CleanOptions.strip_all()` preset removes everything possible - outputs, execution counts, cell metadata, notebook metadata, and kernel info:

In [ ]:
# Strip all (most aggressive)
stripped = nb.clean(CleanOptions.strip_all())
stripped_json = json.loads(stripped.to_string(Format.Ipynb))

print(f"After strip_all:")
print(f"  - kernelspec present: {'kernelspec' in stripped_json.get('metadata', {})}")

## Available CleanOptions

Here are all the available cleaning options:

| Option | Description |
|--------|-------------|
| `remove_outputs` | Remove all outputs from code cells |
| `remove_execution_counts` | Reset execution counts to null |
| `remove_cell_metadata` | Remove cell-level metadata |
| `remove_notebook_metadata` | Remove notebook-level metadata |
| `remove_kernel_info` | Remove kernel specification |
| `preserve_cell_ids` | Keep cell IDs (default: remove) |

And two convenience presets:
- `CleanOptions.for_vcs()` - Ideal for version control
- `CleanOptions.strip_all()` - Remove everything